# exp-019: 독립 LLM 라벨 eval — learnable fusion의 순환성 검증

- **목적:** exp-018의 "learned > arbitrary" 결과가 *규칙 기반 라벨에 head를 맞춘 순환*인지, 아니면 실제 적합도에서도 성립하는지 검증. LLM가 자소서·JD **원문을 직접 읽고** holistic relevance(0~4)를 매긴 독립 라벨로 재평가.
- **차별성 축:** ④ GT 부재 (LLM-API 없이 LLM in-session 판단을 준-GT로) / ⑤ 매칭 head 자체 학습 노선의 신뢰도 점검
- **입력 데이터:** `raw/experiments/exp-019-independent-llm-eval/eval_pairs.csv`(6 held-out user × 10 후보 = 60쌍) + `llm_labels.csv`(LLM holistic 라벨)
- **출력 위치:** `raw/experiments/exp-019-independent-llm-eval/`
- **관련 위키:** exp-018-learnable-fusion-head, learnable-fusion-실험계획
- **작성일:** 2026-05-31 · **시드:** 42 · **LLM 호출:** ❌ 0건 (라벨은 LLM가 세션 내에서 텍스트 읽고 직접 부여)

## 방법

exp-018 held-out user 6명에 대해 후보 10개(role-match 3 / industry-match 2 / competency-match 2 / off-target 3)를 추출, 자소서 STAR·역량과 JD 직무·산업·요구스킬·요약을 텍스트로 덤프했다. **LLM(나)가 60쌍을 직접 읽고** "이 사람에게 이 공고가 얼마나 맞는가"를 0~4로 채점(규칙 컴포넌트와 독립). 라벨은 `llm_labels.csv`에 동결.

In [1]:
import pandas as pd, numpy as np, json
from pathlib import Path
ROOT = Path('.')
OUT = ROOT / 'raw' / 'experiments' / 'exp-019-independent-llm-eval'
COMP_COLS = ['role_match','hard_skill','industry_match','star_overlap','competency']

df = pd.read_csv(OUT / 'eval_pairs.csv')                 # components + arb_E/gold_E/learn_E
lab = pd.read_csv(OUT / 'llm_labels.csv')             # LLM holistic 라벨 (동결)
df = df.merge(lab[['pair_id','llm_label']], on='pair_id')
print(f'{df.userId.nunique()} users × {len(df)} pairs | label dist: {df.llm_label.value_counts().sort_index().to_dict()}')

6 users × 60 pairs | label dist: {0: 25, 1: 14, 2: 14, 3: 3, 4: 4}


In [2]:
# --- 규칙 E-label(exp-018 labeler) vs LLM 독립 라벨: 순환성(겹침) 정도 ---
THRESH=[0.15,0.32,0.52,0.75]
def rule_E(r):
    c=[r.role_match,r.hard_skill,r.industry_match,r.star_overlap,r.competency]
    v=.2*(c[0]+c[1]+c[2]+c[3]+c[4])
    if c[0]==0 and c[2]==0: v*=0.5
    if c[0]==1 and c[2]==1: v=min(1,v+0.1)
    if c[0]==1 and c[1]==0 and c[4]==0: v*=0.8
    return sum(1 for t in THRESH if v>=t)
df['rule_E_label']=df.apply(rule_E,axis=1)
spear=df.rule_E_label.corr(df.llm_label,method='spearman')
print(f'규칙 E-label vs LLM  Spearman={spear:.3f} | 완전일치={ (df.rule_E_label==df.llm_label).mean():.1%} | MAE={(df.rule_E_label-df.llm_label).abs().mean():.2f}')
print('→ 0.75 → 규칙 라벨은 실제 판단과 ~75%만 일치, 25%는 독립적 (순환성 부분적)')

규칙 E-label vs LLM  Spearman=0.754 | 완전일치=58.3% | MAE=0.50
→ 0.75 → 규칙 라벨은 실제 판단과 ~75%만 일치, 25%는 독립적 (순환성 부분적)


In [3]:
# --- 컴포넌트별 실제 적합도(LLM 라벨) 예측력 — 비순환적 신호 검증 ---
print('컴포넌트 vs LLM label Spearman:')
corr=[]
for c in COMP_COLS:
    s=df[c].corr(df.llm_label,method='spearman'); corr.append((c,s))
    print(f'  {c:16s} {s:+.3f}')
print('→ role_match 압도적, competency 최약 (exp-018에서 추가한 신호가 실제 적합도엔 약함)')
pd.DataFrame(corr,columns=['component','spearman_vs_llm']).to_csv(OUT/'component_correlation.csv',index=False)

컴포넌트 vs LLM label Spearman:
  role_match       +0.618
  hard_skill       +0.306
  industry_match   +0.293
  star_overlap     +0.313
  competency       +0.147
→ role_match 압도적, competency 최약 (exp-018에서 추가한 신호가 실제 적합도엔 약함)


In [4]:
# --- NDCG@10: arb_E / gold_E / learn_E 를 LLM 독립 라벨로 평가 ---
def ndcg(scores,labels,k=10):
    o=np.argsort(-scores,kind='stable')[:k]; disc=1/np.log2(np.arange(2,k+2))
    dcg=(labels[o]*disc[:len(o)]).sum(); ideal=np.sort(labels)[::-1][:k]
    idcg=(ideal*disc[:len(ideal)]).sum(); return dcg/idcg if idcg>0 else np.nan
res={'arb_E':[],'gold_E':[],'learn_E':[]}
for uid,g in df.groupby('userId'):
    L=g.llm_label.values.astype(float)
    if L.max()==0: continue
    for m in res: res[m].append(ndcg(g[m].values.astype(float),L))
summary={m:float(np.nanmean(v)) for m,v in res.items()}
print('=== NDCG@10 (LLM 독립 라벨 기준) ===')
for m,v in summary.items(): print(f'  {m:9s} {v:.4f}')
print(f'  learned − arbitrary = {summary["learn_E"]-summary["arb_E"]:+.4f}p  (≤0 → 선형 학습이 독립 평가에선 임의값 대비 이득 없음)')
pd.DataFrame({'metric':list(summary),'ndcg10':list(summary.values())}).to_csv(OUT/'ndcg_independent.csv',index=False)

meta={'experiment':'exp-019-independent-llm-eval','date':'2026-05-31','n_users':int(df.userId.nunique()),'n_pairs':int(len(df)),
 'label_source':'LLM holistic judgment from text (no API, independent of rule labeler)',
 'rule_vs_llm_spearman':float(spear),'rule_llm_exact_agree':float((df.rule_E_label==df.llm_label).mean()),
 'ndcg10_independent':summary,'component_spearman_vs_llm':{c:float(df[c].corr(df.llm_label,method='spearman')) for c in COMP_COLS}}
(OUT/'meta.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2))
print('saved:',sorted(p.name for p in OUT.iterdir()))

=== NDCG@10 (LLM 독립 라벨 기준) ===
  arb_E     0.9165
  gold_E    0.9165
  learn_E   0.9057
  learned − arbitrary = -0.0108p  (≤0 → 선형 학습이 독립 평가에선 임의값 대비 이득 없음)
saved: ['llm_labels.csv', 'component_correlation.csv', 'eval_pairs.csv', 'meta.json', 'ndcg_independent.csv']


## 결과 (요약 — 상세 분석은 위키)

1. **순환성 정량화**: 규칙 E-라벨 vs LLM 독립 라벨 Spearman ≈ **0.75** (완전일치 58%). 규칙 라벨은 실제 판단의 좋은 근사지만 25%는 어긋남 — exp-018의 라벨이 *부분적으로* 순환적이었음을 확인.
2. **role_match가 실제 적합도의 압도적 신호**(ρ≈0.62), competency(ρ≈0.15)·industry·star는 약함. exp-018에서 추가한 competency가 실제 적합도엔 기여가 작음.
3. **learned ≈ arbitrary (독립 라벨)**: NDCG@10 arb 0.92 ≈ gold 0.92 ≥ learn 0.91. exp-018의 learned>arbitrary(+0.062p)는 **규칙 라벨 아티팩트**였고, 독립 평가에선 선형 head가 임의값을 못 이김.

→ **결론**: 5개 규칙 컴포넌트의 선형 fusion만으로는 실제 적합도에서 임의 가중치 이상을 못 낸다. 진짜 이득은 컴포넌트가 못 잡는 **자소서 의미 신호(임베딩, V2)**에서 와야 한다. 차별성 ①(자소서 정성)·②(한국어)를 직접 겨냥.

> 한계: N=6 user / 60쌍, 단일 채점자(LLM). 방향성 신호이며 통계적 결론은 N 확장 후.

## 관련
- exp-018-learnable-fusion-head — 규칙 라벨 기반 결과(이번 실험이 검증)
- learnable-fusion-실험계획 — V2/V3 동기 강화